# Top interactors by interface metrics
Load every `results/<bait>__<prey>/interface_metrics.parquet`, rank preys, and merge with `data/pxd013906_uniprot_mapping.csv`.

In [1]:
from pathlib import Path
import pandas as pd

ROOT = Path("..") if Path("../results").exists() else Path(".")
RANK_METRIC = "ipsae"   # any of: ipsae, ilis, pinc, pdockq2, pair_iptm, ...
TOP_N = 15

In [2]:
files = sorted((ROOT / "results").glob("*/interface_metrics.parquet"))
metrics = pd.concat((pd.read_parquet(f) for f in files), ignore_index=True)
metrics[["bait", "prey"]] = metrics["pair"].str.split("__", n=1, expand=True)
print(len(files), "pairs;", len(metrics), "rows")
metrics.head()

1174 pairs; 1174 rows


,pair,seed,sample_index,cif_path,chain_i,chain_j,lis,clis,ilis,pinc,ipsae,pdockq,pdockq2,pair_iptm,note,bait,prey
0,A1L4W5__A0A1I9LLD2,1,0,A1L4W5__A0A1I9LLD2/raw/sample_0.cif,A,B,0.0,0.0,0.0,0.072391,0.0,0.0990,0.0086,0.110965,,A1L4W5,A0A1I9LLD2
1,A1L4W5__A0A1I9LMI7,1,0,A1L4W5__A0A1I9LMI7/raw/sample_0.cif,A,B,0.0,0.0,0.0,0.061235,0.0,0.0220,0.0081,0.095462,,A1L4W5,A0A1I9LMI7
2,A1L4W5__A0A1I9LPQ2,1,0,A1L4W5__A0A1I9LPQ2/raw/sample_0.cif,A,B,0.0,0.0,0.0,0.183183,0.0,0.0448,0.0103,0.106166,,A1L4W5,A0A1I9LPQ2
3,A1L4W5__A0A1I9LQ33,1,0,A1L4W5__A0A1I9LQ33/raw/sample_0.cif,A,B,0.0,0.0,0.0,0.097983,0.0,0.1747,0.0094,0.106143,,A1L4W5,A0A1I9LQ33
4,A1L4W5__A0A1I9LR66,1,0,A1L4W5__A0A1I9LR66/raw/sample_0.cif,A,B,0.0,0.0,0.0,0.072140,0.0,0.0347,0.0085,0.088038,,A1L4W5,A0A1I9LR66


## Rank preys
Per pair: best value of the ranking metric over diffusion samples and chain pairs. Then take the top N.

In [3]:
per_pair = (metrics.groupby(["pair", "bait", "prey"], as_index=False)
            .agg(best=(RANK_METRIC, "max"),
                 mean=(RANK_METRIC, "mean"),
                 n_rows=(RANK_METRIC, "size"),
                 ilis=("ilis", "max"), pinc=("pinc", "max"),
                 pdockq2=("pdockq2", "max"), pair_iptm=("pair_iptm", "max")))
top = per_pair.sort_values("best", ascending=False).head(TOP_N).reset_index(drop=True)
top

,pair,bait,prey,best,mean,n_rows,ilis,pinc,pdockq2,pair_iptm
0,A1L4W5__O22286,A1L4W5,O22286,0.725252,0.725252,1,0.572864,0.379806,0.0242,0.527265
1,A1L4W5__Q9SRV1,A1L4W5,Q9SRV1,0.724868,0.724868,1,0.583903,0.327081,0.0170,0.512343
2,A1L4W5__Q8L765,A1L4W5,Q8L765,0.721307,0.721307,1,0.603514,0.376467,0.0207,0.509467
3,A1L4W5__A1L4W5,A1L4W5,A1L4W5,0.694379,0.694379,1,0.574443,0.390971,0.0279,0.531382
4,A1L4W5__Q1EBV6,A1L4W5,Q1EBV6,0.690125,0.690125,1,0.602067,0.382738,0.0214,0.496102
5,A1L4W5__Q9M8J9,A1L4W5,Q9M8J9,0.676686,0.676686,1,0.581601,0.360380,0.0225,0.495598
6,A1L4W5__Q9LYA9,A1L4W5,Q9LYA9,0.540445,0.540445,1,0.420300,0.233519,0.0132,0.374568
7,A1L4W5__Q9SEU6,A1L4W5,Q9SEU6,0.526597,0.526597,1,0.576037,0.358660,0.0265,0.385439
8,A1L4W5__Q9SEU8,A1L4W5,Q9SEU8,0.502060,0.502060,1,0.546081,0.311987,0.0215,0.371722
9,A1L4W5__Q93ZE8,A1L4W5,Q93ZE8,0.462267,0.462267,1,0.400061,0.537281,0.0697,0.445629


## Merge with UniProt info table

In [4]:
info = pd.read_csv(ROOT / "data" / "pxd013906_uniprot_mapping.csv")
info = info.drop_duplicates("uniprot")   # one row per UniProt id
merged = top.merge(info, left_on="prey", right_on="uniprot", how="left")
print("unmatched preys:", merged["uniprot"].isna().sum())
merged

unmatched preys: 0


,pair,bait,prey,best,mean,n_rows,ilis,pinc,pdockq2,pair_iptm,...,uniprot,reviewed,genes,proteinDescription,length,organism,logfc,pval,adjp,annotation
0,A1L4W5__O22286,A1L4W5,O22286,0.725252,0.725252,1,0.572864,0.379806,0.0242,0.527265,...,O22286,True,BPM3,BTB/POZ and MATH domain-containing protein 3,408.0,Arabidopsis thaliana,5.392388,5.398775e-05,9.532030e-04,NaN
1,A1L4W5__Q9SRV1,A1L4W5,Q9SRV1,0.724868,0.724868,1,0.583903,0.327081,0.0170,0.512343,...,Q9SRV1,True,BPM4,BTB/POZ and MATH domain-containing protein 4,465.0,Arabidopsis thaliana,7.180554,3.989501e-12,2.519523e-10,BPM4
2,A1L4W5__Q8L765,A1L4W5,Q8L765,0.721307,0.721307,1,0.603514,0.376467,0.0207,0.509467,...,Q8L765,True,BPM1,BTB/POZ and MATH domain-containing protein 1,407.0,Arabidopsis thaliana,5.931116,3.277445e-06,7.914066e-05,BPM1
3,A1L4W5__A1L4W5,A1L4W5,A1L4W5,0.694379,0.694379,1,0.574443,0.390971,0.0279,0.531382,...,A1L4W5,True,BPM6,BTB/POZ and MATH domain-containing protein 6,415.0,Arabidopsis thaliana,8.563669,4.754271e-33,1.301085e-30,NaN
4,A1L4W5__Q1EBV6,A1L4W5,Q1EBV6,0.690125,0.690125,1,0.602067,0.382738,0.0214,0.496102,...,Q1EBV6,True,BPM5,BTB/POZ and MATH domain-containing protein 5,410.0,Arabidopsis thaliana,8.801948,6.753904e-22,7.393273e-20,BPM5
5,A1L4W5__Q9M8J9,A1L4W5,Q9M8J9,0.676686,0.676686,1,0.581601,0.360380,0.0225,0.495598,...,Q9M8J9,True,BPM2,BTB/POZ and MATH domain-containing protein 2,406.0,Arabidopsis thaliana,6.163266,6.283918e-07,1.778999e-05,BPM2
6,A1L4W5__Q9LYA9,A1L4W5,Q9LYA9,0.540445,0.540445,1,0.420300,0.233519,0.0132,0.374568,...,Q9LYA9,True,CSP41A,Chloroplast stem-loop binding protein of 41 kD...,406.0,Arabidopsis thaliana,5.587845,1.734651e-05,3.605438e-04,NaN
7,A1L4W5__Q9SEU6,A1L4W5,Q9SEU6,0.526597,0.526597,1,0.576037,0.358660,0.0265,0.385439,...,Q9SEU6,True,NaN,"Thioredoxin M4, chloroplastic",193.0,Arabidopsis thaliana,1.111247,4.506007e-01,4.748950e-01,NaN
8,A1L4W5__Q9SEU8,A1L4W5,Q9SEU8,0.502060,0.502060,1,0.546081,0.311987,0.0215,0.371722,...,Q9SEU8,True,TRXM2,"Thioredoxin M2, chloroplastic",186.0,Arabidopsis thaliana,1.111247,4.506007e-01,4.748950e-01,NaN
9,A1L4W5__Q93ZE8,A1L4W5,Q93ZE8,0.462267,0.462267,1,0.400061,0.537281,0.0697,0.445629,...,Q93ZE8,True,SDF2,Stromal cell-derived factor 2-like protein,218.0,Arabidopsis thaliana,1.150096,4.451117e-01,4.748950e-01,NaN


In [ ]:
merged